# 16. 迭代器、生成器和装饰器

这一章学习 Python 中比较重要的进阶语法：

- 可迭代对象和迭代器
- `iter()` 和 `next()`
- 自定义迭代器
- 生成器函数和 `yield`
- 生成器表达式
- `yield from`
- 装饰器基础
- 带参数的装饰器
- 多个装饰器叠加

这三个主题都和“函数可以被传递、代码可以被延迟执行”有关。


## 1. 可迭代对象和迭代器

能被 `for` 循环遍历的对象叫可迭代对象，例如列表、元组、字符串、字典、集合、文件对象。

迭代器是可以用 `next()` 一次取出一个值的对象。


In [1]:
nums = [10, 20, 30]

# iter() 可以把可迭代对象转换成迭代器
iterator = iter(nums)

print(next(iterator))
print(next(iterator))
print(next(iterator))

# print(next(iterator))
# 如果继续调用 next()，没有值可取时会抛出 StopIteration。


10
20
30


### 解释

- 可迭代对象不一定是迭代器。
- 迭代器会记住当前取到哪里了。
- `for` 循环背后其实就是不断调用 `next()`。
- 当迭代器没有下一个值时，会抛出 `StopIteration`，`for` 循环会自动处理它。


## 2. 自定义迭代器

一个对象如果实现了 `__iter__()` 和 `__next__()`，就可以作为迭代器使用。


In [2]:
class CountDown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        # 迭代器对象的 __iter__ 通常返回 self
        return self

    def __next__(self):
        if self.current <= 0:
            # 没有更多数据时，必须抛出 StopIteration
            raise StopIteration

        value = self.current
        self.current -= 1
        return value


for num in CountDown(5):
    print(num)


5
4
3
2
1


### 解释

- `__iter__()` 返回迭代器对象。
- `__next__()` 返回下一个值。
- 自定义迭代器适合控制复杂的取值过程。
- 简单场景通常用生成器更方便。


## 3. 生成器函数和 `yield`

函数中只要出现 `yield`，这个函数就不是普通函数，而是生成器函数。


In [3]:
def count_up_to(n):
    current = 1

    while current <= n:
        # yield 会产出一个值，并暂停函数执行
        yield current
        current += 1


gen = count_up_to(3)

print(next(gen))
print(next(gen))
print(next(gen))

print('使用 for 循环遍历生成器：')
for num in count_up_to(5):
    print(num)


1
2
3
使用 for 循环遍历生成器：
1
2
3
4
5


### 解释

- 普通函数执行后返回一个结果。
- 生成器可以分多次产出结果，每次遇到 `yield` 暂停。
- 下一次调用 `next()` 时，会从上一次暂停的位置继续执行。
- 生成器适合处理大量数据，因为它可以按需生成，不必一次性创建完整列表。


## 4. 生成器表达式

生成器表达式和列表推导式很像，但它不会一次性生成完整列表，而是按需产生数据。


In [4]:
nums = [1, 2, 3, 4, 5]

list_result = [x * x for x in nums]
generator_result = (x * x for x in nums)

print(list_result)
print(generator_result)

for value in generator_result:
    print(value)


[1, 4, 9, 16, 25]
<generator object <genexpr> at 0x00000262CEE2D220>
1
4
9
16
25


### 解释

- `[x * x for x in nums]` 得到列表。
- `(x * x for x in nums)` 得到生成器。
- 生成器只能按顺序消费一次。
- 数据量很大时，生成器表达式更节省内存。


## 5. `yield from`

`yield from` 可以把另一个可迭代对象中的值逐个产出，让生成器代码更简洁。


In [5]:
def flatten(matrix):
    for row in matrix:
        # 等价于：
        # for item in row:
        #     yield item
        yield from row


matrix = [
    [1, 2, 3],
    [4, 5],
    [6, 7, 8]
]

print(list(flatten(matrix)))


[1, 2, 3, 4, 5, 6, 7, 8]


### 解释

- `yield from row` 会把 `row` 中的元素一个个交出去。
- 它常用于展开嵌套结构，或把生成任务委托给另一个生成器。


## 6. 装饰器基础

装饰器本质上是一个函数：接收一个函数，返回一个新函数。

它常用于在不修改原函数代码的情况下，给函数增加额外功能。


In [6]:
def log_decorator(func):
    def wrapper():
        print('函数开始执行')
        func()
        print('函数执行结束')

    return wrapper


@log_decorator
def say_hello():
    print('hello')


say_hello()


函数开始执行
hello
函数执行结束


### 解释

- `@log_decorator` 等价于 `say_hello = log_decorator(say_hello)`。
- 原函数被包装成了 `wrapper`。
- 调用 `say_hello()` 时，实际执行的是包装后的函数。


## 7. 支持参数和返回值的装饰器

真实项目中的函数通常有参数和返回值，因此装饰器需要使用 `*args`、`**kwargs` 转发参数。


In [7]:
from functools import wraps


def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f'准备调用函数：{func.__name__}')
        result = func(*args, **kwargs)
        print(f'函数调用结束：{func.__name__}')
        return result

    return wrapper


@log_call
def add(a, b):
    return a + b


result = add(10, 20)
print('结果：', result)
print('函数名：', add.__name__)


准备调用函数：add
函数调用结束：add
结果： 30
函数名： add


### 解释

- `*args` 接收任意位置参数。
- `**kwargs` 接收任意关键字参数。
- `return result` 把原函数的返回值继续返回给调用者。
- `@wraps(func)` 可以保留原函数的名称、文档字符串等信息。


## 8. 带参数的装饰器

如果装饰器本身也需要参数，就需要再包一层函数。


In [8]:
from functools import wraps


def repeat(times):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(times):
                result = func(*args, **kwargs)
            return result

        return wrapper

    return decorator


@repeat(3)
def greet(name):
    print(f'你好，{name}')


greet('小明')


你好，小明
你好，小明
你好，小明


### 解释

- `repeat(3)` 先执行，返回真正的装饰器 `decorator`。
- `decorator` 接收原函数 `func`。
- `wrapper` 才是最终调用时执行的新函数。


## 9. 装饰器应用：计时

装饰器常用于日志、权限校验、缓存、计时、重试等场景。


In [9]:
from functools import wraps
from time import perf_counter, sleep


def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        end = perf_counter()
        print(f'{func.__name__} 耗时：{end - start:.4f} 秒')
        return result

    return wrapper


@timer
def slow_task():
    sleep(0.2)
    return '完成'


print(slow_task())


slow_task 耗时：0.2003 秒
完成


### 解释

- `perf_counter()` 适合做短时间性能计时。
- 装饰器让计时代码和业务代码分离。
- 多个函数都需要计时时，可以复用同一个装饰器。


## 10. 小练习：综合示例

1. 写一个生成器，产生 1 到 n 之间的偶数。
2. 写一个装饰器，调用函数前打印函数名。
3. 用装饰器包装一个求和函数。


In [10]:
from functools import wraps


def even_numbers(n):
    for num in range(1, n + 1):
        if num % 2 == 0:
            yield num


def show_name(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print('正在调用：', func.__name__)
        return func(*args, **kwargs)

    return wrapper


@show_name
def sum_range(n):
    return sum(range(1, n + 1))


print(list(even_numbers(10)))
print(sum_range(100))


[2, 4, 6, 8, 10]
正在调用： sum_range
5050


## 11. 常见错误总结

1. 生成器被消费一次后，再遍历就没有数据了。
2. 自定义迭代器忘记在结束时抛出 `StopIteration`。
3. 把生成器当列表使用，例如直接用下标访问。
4. 装饰器中忘记返回 `wrapper`。
5. `wrapper` 中忘记调用原函数。
6. 装饰器没有返回原函数结果，导致调用者拿到 `None`。
7. 装饰器没有使用 `*args`、`**kwargs`，导致无法包装带参数函数。
8. 多层装饰器执行顺序理解错误。
